# 03 — Inference Demo

Runs the trained YOLOv8 person detector on validation images, custom uploads, and video.

| | |
|---|---|
| **Model** | YOLOv8s fine-tuned on merged person dataset |
| **Classes** | `0: person` |
| **Weights** | `best.pt` from 02_train.ipynb |
| **Conf threshold** | 0.5 |

In [ ]:
!pip install ultralytics pyyaml opencv-python-headless matplotlib -q

import os
import shutil
import glob
import random
import yaml
import cv2
import base64
import numpy as np
import matplotlib.pyplot as plt
import ultralytics
from ultralytics import YOLO
from ultralytics.utils import SETTINGS
from IPython.display import HTML, display
from google.colab import files

print('Ultralytics version:', ultralytics.__version__)
print('OpenCV version:', cv2.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/AI_TRAINING/GreenVision'
os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
import torch

# --- Config ---
DRIVE_ZIP = os.path.join(DRIVE_ROOT, 'merged_dataset.zip')
DATA_YAML = '/content/data.yaml'
DATASET_DIR = '/content/dataset/merged'
CONF = 0.5
LOCAL_BEST = '/content/best.pt'

# Auto-detect GPU
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

# Unzip merged dataset — always re-unzip to ensure complete
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError('merged_dataset.zip not found. Run 01d first.')
print('Unzipping merged dataset...')
!unzip -q -o {DRIVE_ZIP} -d /content/dataset
print('Done.')

# Write data.yaml
config = {
    'path'  : DATASET_DIR,
    'train' : 'images/train',
    'val'   : 'images/val',
    'nc'    : 1,
    'names' : {0: 'person'},
}
with open(DATA_YAML, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)
SETTINGS.update({'datasets_dir': '/content/dataset'})

# Verify
val_imgs = len(glob.glob(os.path.join(DATASET_DIR, 'images', 'val', '*.*')))
print(f'Val images: {val_imgs}')

# Find best.pt: check models/ first, then runs/
DRIVE_BEST = None
candidates = [os.path.join(DRIVE_ROOT, 'models', 'best.pt')]
runs_dir = os.path.join(DRIVE_ROOT, 'runs')
if os.path.exists(runs_dir):
    for rd in sorted(glob.glob(os.path.join(runs_dir, 'person_detect_v1*'))):
        candidates.append(os.path.join(rd, 'weights', 'best.pt'))

for c in candidates:
    if os.path.exists(c):
        DRIVE_BEST = c
        break

if DRIVE_BEST is None:
    raise FileNotFoundError('best.pt not found. Run 02_train.ipynb first.')

print(f'Found model: {DRIVE_BEST}')

# Copy best.pt local
if not os.path.exists(LOCAL_BEST):
    shutil.copy2(DRIVE_BEST, LOCAL_BEST)
    print(f'Copied best.pt ({os.path.getsize(LOCAL_BEST)/1e6:.1f} MB)')
else:
    print(f'best.pt already local ({os.path.getsize(LOCAL_BEST)/1e6:.1f} MB)')

# Load model
model = YOLO(LOCAL_BEST)
device = next(model.model.parameters()).device
print(f'Model loaded on: {device}')
print(f'Class names: {model.names}')

## 1. Run on Validation Set

Evaluates the model on the merged dataset's validation split and prints mAP, precision, and recall.

In [ ]:
# Verify data.yaml and dataset
print('data.yaml:', DATA_YAML)
print(open(DATA_YAML).read())

val_img_dir = os.path.join(DATASET_DIR, 'images', 'val')
val_count = len(glob.glob(os.path.join(val_img_dir, '*.*')))
print(f'Validation images: {val_count}')

In [ ]:
metrics = model.val(data=DATA_YAML, conf=CONF, device=DEVICE)

print('\n' + '='*40)
print('  Validation Metrics')
print('='*40)
print(f'  mAP@0.5      : {metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'  Precision    : {metrics.box.mp:.4f}')
print(f'  Recall       : {metrics.box.mr:.4f}')
print('='*40)

## 2. Predict on Sample Images

Picks 12 random validation images, runs detection, and displays a 3x4 grid with bounding boxes in green and person counts.

In [ ]:
# Collect validation images
val_images = glob.glob(os.path.join(val_img_dir, '*.jpg')) + \
             glob.glob(os.path.join(val_img_dir, '*.png'))
print(f'Found {len(val_images)} validation images')

# Pick 12 random images
random.seed(42)
sample_imgs = random.sample(val_images, min(12, len(val_images)))
print(f'Selected {len(sample_imgs)} sample images')

# Run prediction
results = []
for img_path in sample_imgs:
    result = model.predict(img_path, conf=CONF, classes=[0], verbose=False)[0]
    results.append((img_path, result))

print(f'Predictions done for {len(results)} images')

In [ ]:
# Draw 3x4 grid
ncols = 4
nrows = (len(results) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = axes.flatten() if len(results) > 1 else [axes]

GREEN = (0, 255, 0)

for idx, (img_path, result) in enumerate(results):
    # Read original image
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Draw bounding boxes
    person_count = 0
    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = float(box.conf[0])
            cv2.rectangle(img, (x1, y1), (x2, y2), GREEN, 2)
            cv2.putText(img, f'person {conf:.2f}', (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, GREEN, 2)
            person_count += 1

    # Title with person count
    axes[idx].imshow(img)
    axes[idx].set_title(f'{person_count} person(s)', fontsize=12)
    axes[idx].axis('off')

# Hide unused axes
for idx in range(len(results), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 3. Predict on Custom Image

Upload your own image and see detection results.

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f'Uploaded: {filename} ({len(uploaded[filename])/1e3:.1f} KB)')

result = model.predict(filename, conf=CONF, classes=[0], verbose=False)[0]

# Draw results
img = cv2.imread(filename)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

person_count = 0
if result.boxes is not None and len(result.boxes) > 0:
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        conf = float(box.conf[0])
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), GREEN, 2)
        cv2.putText(img_rgb, f'person {conf:.2f}', (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, GREEN, 2)
        person_count += 1

plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)
plt.title(f'Detected {person_count} person(s)', fontsize=14)
plt.axis('off')
plt.show()

## 4. Predict on Video

Upload a video file. The notebook processes it frame-by-frame, draws bounding boxes, and saves the output.

After processing, stats are printed: total frames, frames with people detected, max people in a single frame.

In [ ]:
uploaded = files.upload()
video_filename = list(uploaded.keys())[0]
print(f'Uploaded: {video_filename} ({os.path.getsize(video_filename)/1e6:.1f} MB)')

In [ ]:
# Process video frame-by-frame
OUTPUT_RAW = '/content/output_raw.avi'
OUTPUT_MP4 = '/content/output.mp4'

os.makedirs(os.path.dirname(OUTPUT_RAW), exist_ok=True)

cap = cv2.VideoCapture(video_filename)
if not cap.isOpened():
    raise RuntimeError(f'Cannot open video: {video_filename}')

fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f'Video: {width}x{height} @ {fps:.1f} FPS, {total_frames} frames')

fourcc = cv2.VideoWriter_fourcc(*'XVID')
writer = cv2.VideoWriter(OUTPUT_RAW, fourcc, fps, (width, height))

frames_with_people = 0
max_people = 0
frame_idx = 0
# Store sample frames for later display
sample_frame_indices = set()
if total_frames > 0:
    sample_positions = [int(i * (total_frames - 1) / 4) for i in range(5)]
    sample_frame_indices = set(sample_positions)
sample_frames = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_idx += 1
    if frame_idx % 100 == 0:
        print(f'  Processing frame {frame_idx}/{total_frames}...')

    # Run detection
    result = model.predict(frame, conf=CONF, classes=[0], verbose=False)[0]

    person_count = 0
    if result.boxes is not None and len(result.boxes) > 0:
        person_count = len(result.boxes)
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            conf = float(box.conf[0])
            cv2.rectangle(frame, (x1, y1), (x2, y2), GREEN, 2)
            cv2.putText(frame, f'person {conf:.2f}', (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, GREEN, 2)

    if person_count > 0:
        frames_with_people += 1
    max_people = max(max_people, person_count)

    # Store sample frame
    if (frame_idx - 1) in sample_frame_indices:
        sample_frames.append((frame_idx, frame.copy(), person_count))

    writer.write(frame)

cap.release()
writer.release()

print(f'\nDone! Processed {frame_idx} frames.')
print(f'  Frames with people : {frames_with_people}')
print(f'  Max people in frame: {max_people}')

# Re-encode to H.264 MP4 for browser playback
print('\nRe-encoding to H.264...')
!ffmpeg -y -i {OUTPUT_RAW} -c:v libx264 -preset fast -crf 23 -an {OUTPUT_MP4} 2>/dev/null
print(f'Output saved: {OUTPUT_MP4} ({os.path.getsize(OUTPUT_MP4)/1e6:.1f} MB)')

## 5. Sample Video Frames

Displays 5 evenly-spaced frames from the processed video with detection boxes.

In [ ]:
if len(sample_frames) == 0:
    print('No sample frames captured.')
else:
    ncols = min(5, len(sample_frames))
    nrows = 1
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4))
    if ncols == 1:
        axes = [axes]

    for idx, (fnum, frame, count) in enumerate(sample_frames):
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(img_rgb)
        axes[idx].set_title(f'Frame {fnum} ({count} person(s))', fontsize=10)
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Show inline video player
output_size = os.path.getsize(OUTPUT_MP4)

if output_size > 200 * 1024 * 1024:
    print(f'Output video is {output_size/1e6:.1f} MB — too large for inline playback.')
    print(f'Saved at: {OUTPUT_MP4}')
    print('Download it or view in the Files panel on the left.')
else:
    with open(OUTPUT_MP4, 'rb') as f:
        video_b64 = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video width="640" controls '
        f'src="data:video/mp4;base64,{video_b64}"></video>'
    ))

## 6. Save to Drive

Copies the output video to the demo_outputs folder on Google Drive.

In [ ]:
demo_dir = os.path.join(DRIVE_ROOT, 'demo_outputs')
os.makedirs(demo_dir, exist_ok=True)

dst_video = os.path.join(demo_dir, 'inference_output.mp4')
shutil.copy2(OUTPUT_MP4, dst_video)
print(f'Output video saved to: {dst_video}')
print(f'Size: {os.path.getsize(dst_video)/1e6:.1f} MB')

---
## Done!

Inference notebook completed. Outputs:

- Validation metrics printed above
- 12 sample predictions with bounding boxes
- Custom image detection
- Processed video saved to:
  `/content/drive/MyDrive/AI_TRAINING/GreenVision/demo_outputs/inference_output.mp4`